# MojoVec Quickstart

This notebook is a complete tour of the managed MojoVec Python API:

- Flat Float32 and SQ8 HNSW collections;
- batch CRUD with metadata and documents;
- vector search and Chroma-style metadata filters;
- BM25 full-text search and hybrid RRF;
- statistics, soft deletion, and compaction;
- optional NumPy zero-copy paths;
- atomic snapshots, memory-mapped loading, and WAL recovery.

MojoVec runs in-process and owns all native resources. There are no servers, raw pointers, or manual memory-release calls.

## 1. Install MojoVec

Install the current PyPI release into the environment used by this notebook kernel. Restart the kernel once after upgrading an already imported version.

In [ ]:
%pip install mojovec

In [ ]:
import mojovec

In [ ]:
print("MojoVec version:", mojovec.__version__)
print("Native backend:", mojovec.native_backend())
print("Default mmap threshold:", mojovec.DEFAULT_MMAP_THRESHOLD_BYTES)

# Python wrappers expose normal runtime documentation and signatures.
help(mojovec.Collection.query)

## 2. Create an SQ8 collection

Set `quantized=True` for compact SQ8 vectors or `False` for exact Float32 storage. The remaining API is identical.

Supported metrics are `l2`, `cosine`, and `ip`. Public distances are always smaller-is-better.

In [ ]:
collection = mojovec.Collection(
    dimension=4,
    M=16,
    ef_construction=96,
    ef_search=64,
    quantized=True,
    metric="cosine",
    name="knowledge_base",
)

print(collection)
print("storage:", collection.storage_kind())
print("metric:", collection.metric())

## 3. Prepare a small batch

IDs are ordinary Python integers. Metadata values may be `str`, `int`, `float`, or `bool`; documents are strings.

In [ ]:
ids = [101, 202, 303, 404, 505]
embeddings = [
    [1.0, 0.0, 0.0, 0.0],
    [0.9, 0.1, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.9, 0.1],
]
metadatas = [
    {"category": "guide", "year": 2024, "published": True},
    {"category": "internals", "year": 2026, "published": True},
    {"category": "release", "year": 2026, "published": False},
    {"category": "tutorial", "year": 2027, "published": True},
    {"category": "guide", "year": 2028, "published": True},
]
documents = [
    "Vector search introduction",
    "HNSW graph traversal and vector search",
    "Database release notes",
    "BM25 full text search tutorial",
    "Persistence snapshots and memory mapping guide",
]

print("records prepared:", len(ids))

## 4. Batch add vectors, metadata, and documents

`add` accepts only new IDs. Use `upsert` when a batch may contain both new and existing IDs.

In [ ]:
collection.add(
    ids=ids,
    embeddings=embeddings,
    metadatas=metadatas,
    documents=documents,
)

print("stats:", collection.stats())
print("metadata 202:", collection.get_metadata(202))
print("document 202:", collection.get_document(202))

## 5. Inspect managed query results

Every query returns the aligned keys `ids`, `distances`, `metadatas`, `documents`, and `scores`.

In [ ]:
def show_results(result):
    for query_index, row_ids in enumerate(result["ids"]):
        print(f"query {query_index}")
        values = result["distances"] or result["scores"]
        for rank, record_id in enumerate(row_ids):
            if record_id < 0:
                continue
            print(
                f"  rank={rank + 1} id={record_id} value={values[query_index][rank]:.6f}\n"
                f"    metadata={result['metadatas'][query_index][rank]}\n"
                f"    document={result['documents'][query_index][rank]}"
            )

## 6. Vector search with a metadata filter

`where` supports `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, `$and`, `$or`, and `$not`.

In [ ]:
vector_result = collection.query(
    query_embeddings=[[1.0, 0.0, 0.0, 0.0]],
    n_results=4,
    where={
        "$and": [
            {"published": True},
            {"year": {"$gte": 2024}},
            {"category": {"$in": ["guide", "internals", "tutorial"]}},
        ]
    },
)

show_results(vector_result)

## 7. Batched vector queries

Nested rows execute multiple searches in one managed API call.

In [ ]:
batch_result = collection.query(
    query_embeddings=[
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
    ],
    n_results=2,
)

show_results(batch_result)

## 8. BM25 document search

Passing `query_texts` selects BM25. The analyzer applies Unicode lowercase conversion, word boundaries, and built-in English and Russian stopwords.

In [ ]:
bm25_result = collection.query(
    query_texts=["HNSW vector search", "memory mapping guide"],
    n_results=3,
    where={"published": True},
)

show_results(bm25_result)

## 9. Hybrid vector + text retrieval

Hybrid search fuses HNSW and BM25 rankings with reciprocal rank fusion. Each vector and text at the same batch position form one query.

In [ ]:
hybrid_result = collection.query_hybrid(
    query_embeddings=[[1.0, 0.0, 0.0, 0.0]],
    query_texts=["HNSW graph traversal"],
    n_results=3,
    rrf_k=60,
    candidate_multiplier=4,
    where={"published": True},
)

show_results(hybrid_result)

## 10. Update, upsert, and delete

- `update` requires every ID to exist;
- `upsert` inserts missing IDs and replaces existing records;
- `delete` performs idempotent soft deletion;
- vector-only replacements preserve metadata and documents.

In [ ]:
collection.update(
    ids=[101],
    embeddings=[[0.95, 0.05, 0.0, 0.0]],
    metadatas=[{"category": "guide", "year": 2029, "published": True}],
    documents=["Updated vector search introduction"],
)

collection.upsert(
    ids=[202, 606],
    embeddings=[
        [0.85, 0.15, 0.0, 0.0],
        [0.0, 0.0, 0.8, 0.2],
    ],
    metadatas=[
        {"category": "internals", "year": 2030, "published": True},
        {"category": "operations", "year": 2030, "published": True},
    ],
    documents=[
        "Updated HNSW implementation notes",
        "WAL recovery and checkpoint operations",
    ],
)

collection.delete([303, 999999])  # Unknown IDs are ignored.
print("after mutations:", collection.stats())

## 11. Reclaim deleted graph rows

Updates and upserts append new rows and soft-delete old versions. `compact_if_needed` rebuilds only when the deleted ratio reaches the chosen threshold.

In [ ]:
before_compaction = collection.stats()
report = collection.compact_if_needed(deleted_ratio=0.20)
after_compaction = collection.stats()

print("before:", before_compaction)
print("report:", report)
print("after:", after_compaction)

## 12. Use the same API for Flat and SQ8

This small comparison demonstrates API parity. It is not a meaningful performance benchmark.

In [ ]:
flat_collection = mojovec.Collection(
    dimension=4,
    M=16,
    ef_construction=96,
    ef_search=64,
    quantized=False,
    metric="cosine",
    name="flat_copy",
)
flat_collection.add(ids, embeddings, metadatas, documents)

sq8_neighbors = collection.query([[1.0, 0.0, 0.0, 0.0]], n_results=3)
flat_neighbors = flat_collection.query([[1.0, 0.0, 0.0, 0.0]], n_results=3)

print("SQ8 storage:", collection.storage_kind(), sq8_neighbors["ids"])
print("Flat storage:", flat_collection.storage_kind(), flat_neighbors["ids"])

## 13. L2, cosine, and inner-product metrics

Metric selection is independent of Flat versus SQ8 storage.

In [ ]:
for metric in ("l2", "cosine", "ip"):
    metric_collection = mojovec.Collection(
        dimension=2,
        quantized=False,
        metric=metric,
        name=f"{metric}_example",
    )
    metric_collection.add(
        ids=[1, 2, 3],
        embeddings=[
            [1.0, 0.0],
            [0.8, 0.2],
            [0.0, 1.0],
        ],
    )
    result = metric_collection.query([[1.0, 0.0]], n_results=3)
    print(metric, result["ids"][0], result["distances"][0])

## 14. Optional NumPy zero-copy fast path

Contiguous `int64` IDs and `float32` embeddings can cross the Python/native boundary without conversion. This section skips itself when NumPy is unavailable.

In [ ]:
try:
    import numpy as np
except ImportError:
    print("NumPy is not installed; skipping the optional fast-path example.")
else:
    numpy_collection = mojovec.Collection(
        dimension=4,
        quantized=True,
        metric="cosine",
        name="numpy_fast_path",
    )
    numpy_ids = np.asarray([701, 702, 703], dtype=np.int64)
    numpy_embeddings = np.ascontiguousarray(
        [
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
        ],
        dtype=np.float32,
    )
    numpy_collection.upsert_numpy(numpy_ids, numpy_embeddings)
    numpy_query = np.ascontiguousarray([[1.0, 0.0, 0.0, 0.0]], dtype=np.float32)
    print(numpy_collection.query_numpy(numpy_query, n_results=2))

## 15. Atomic save, snapshots, and memory-mapped loading

The collection owns mappings and file handles. Mutating a mapped collection transparently materializes writable arrays.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporary_directory = TemporaryDirectory(prefix="mojovec-quickstart-")
data_directory = Path(temporary_directory.name)
database_path = data_directory / "knowledge_base.mojovec"
snapshot_path = data_directory / "point_in_time.mojovec"

collection.save(database_path)
loaded = mojovec.load(
    database_path,
    memory_mapped=True,
    mmap_threshold_bytes=0,
)
snapshot_view = collection.snapshot(
    snapshot_path,
    memory_mapped=True,
    mmap_threshold_bytes=0,
)

print("loaded memory mapped:", loaded.is_memory_mapped())
print("snapshot memory mapped:", snapshot_view.is_memory_mapped())
print("loaded stats:", loaded.stats())
show_results(loaded.query([[1.0, 0.0, 0.0, 0.0]], n_results=2))

## 16. Optional WAL recovery

A snapshot is the recovery base. Mutations appended afterward can be replayed from the WAL after a restart.

In [ ]:
wal_snapshot_path = data_directory / "durable_base.mojovec"
wal_path = data_directory / "durable_changes.wal"

durable = mojovec.Collection(
    dimension=2,
    quantized=False,
    metric="l2",
    name="durable",
)
durable.add([1], [[1.0, 0.0]], documents=["Base snapshot record"])
durable.save(wal_snapshot_path)

durable.enable_wal(wal_path, durability=mojovec.WAL_SYNC)
durable.upsert(
    [2],
    [[0.0, 1.0]],
    documents=["Record committed through the WAL"],
)
durable.flush_wal()

recovered = mojovec.recover(
    wal_snapshot_path,
    wal_path,
    durability=mojovec.WAL_SYNC,
    memory_mapped=True,
    mmap_threshold_bytes=0,
)

print("WAL enabled:", recovered.wal_enabled())
print("WAL sequence:", recovered.wal_sequence())
print("recovered count:", recovered.count())
print("recovered document:", recovered.get_document(2))

## 17. Clean up temporary files

The collections own native mappings. Removing the Python references and temporary directory is enough; there is no native `free()` API.

In [ ]:
del recovered, durable, snapshot_view, loaded
temporary_directory.cleanup()
print("Temporary quickstart files removed.")

## Next steps

- Increase `ef_search` to trade query latency for recall.
- Tune `M` and `ef_construction` on representative data.
- Use `quantized=True` for SQ8 bandwidth savings and `False` for Flat Float32 storage.
- Run the dedicated scripts in `examples/python/` for focused tutorials.
- Use the cold benchmark suite in `benchmarks/cold_bench/` for controlled performance measurements.